# FreshGuard — Notebook 5 / 5: End‑to‑End Evaluation

**Purpose.** Produce the headline numbers that go into the model card. Three independent blocks, all evaluated on the **dedup'd test split** (cluster‑disjoint from train and val):

1. **Classifier on ground‑truth full images** — isolates classifier accuracy from detector recall. 24‑way macro F1, top‑1, per‑class precision/recall/F1.
2. **Detector across operating points** — runs YOLO26s at `conf ∈ {0.15, 0.25, 0.35}` so the model card can document a recall/precision tradeoff curve.
3. **End‑to‑end pipeline (the actual user experience)** — replays the local `FreshnessPipeline.predict` logic: detector first, classifier as fallback when zero detections, `unknown` when both fail.

**Inputs:**
- `ulnnproject/food-freshness-dataset` — raw images
- `<your-handle>/freshguard-manifest`
- `<your-handle>/freshguard-yolo26s`
- `<your-handle>/freshguard-efficientnetv2s`

**Outputs (committed back to the local repo via copy‑paste from this notebook):**
- `eval_report.json` — machine‑readable metrics for all three blocks
- `eval_report.md` — human‑readable summary; paste into `PRD.md` and `src/freshness/ui/content.py`

In [ ]:
!pip install --quiet "timm>=1.0.0" "ultralytics>=8.3.0,<9" "scikit-learn>=1.3" pyarrow

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import torch
from PIL import Image, ImageFile
from sklearn.metrics import classification_report, f1_score
from timm.data import create_transform
from torch.utils.data import DataLoader, Dataset
from ultralytics import YOLO

ImageFile.LOAD_TRUNCATED_IMAGES = False

# ---------------------------------------------------------------
# Configuration — paths auto-resolved so Kaggle's nested layout
# (e.g. freshguard-efficientnetv2s/freshguard-efficientnetv2s/*.pt)
# is tolerated alongside the flat layout.
# ---------------------------------------------------------------
def _resolve_one(roots: list[Path], pattern: str) -> Path:
    candidates: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        candidates.extend(root.rglob(pattern))
    if not candidates:
        raise FileNotFoundError(f"No file matching {pattern!r} under {roots}")
    candidates.sort(key=lambda p: len(p.parts))
    return candidates[0]

MANIFEST_PATH = _resolve_one(
    [Path("/kaggle/input/datasets/elsmmany/freshguard-manifest")],
    "manifest.parquet",
)
DETECTOR_DATASET_YAML = _resolve_one(
    [Path("/kaggle/input/datasets/elsmmany/freshguard-detector-data")],
    "detector_dataset.yaml",
)
DETECTOR_CKPT = _resolve_one(
    [Path("/kaggle/input/datasets/elsmmany/freshguard-yolo26s")],
    "yolo26s_food_freshness.pt",
)
CLASSIFIER_CKPT = _resolve_one(
    [Path("/kaggle/input/datasets/elsmmany/freshguard-efficientnetv2s")],
    "efficientnetv2s_food_freshness.pt",
)
OUTPUT_DIR = Path("/kaggle/working")

DETECTOR_OPERATING_POINTS = (0.15, 0.25, 0.35)
FALLBACK_THRESHOLD = 0.4   # mirror configs/inference.toml in the local repo
MAX_DETECTIONS = 8
BATCH_SIZE = 64
NUM_WORKERS = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device:     {device}")
print(f"Manifest:   {MANIFEST_PATH}")
print(f"YAML:       {DETECTOR_DATASET_YAML}")
print(f"Detector:   {DETECTOR_CKPT}")
print(f"Classifier: {CLASSIFIER_CKPT}")

In [ ]:
PRODUCE_TYPES = (
    "apple", "banana", "bellpepper", "bitter_gourd", "carrot", "cucumber",
    "mango", "okra", "orange", "potato", "strawberry", "tomato",
)
FRESHNESS_LEVELS = ("fresh", "rotten")
COMBINED_CLASSES = tuple(f"{t}_{f}" for t in PRODUCE_TYPES for f in FRESHNESS_LEVELS)
CLASS_TO_INDEX = {c: i for i, c in enumerate(COMBINED_CLASSES)}
NUM_CLASSES = len(COMBINED_CLASSES)

manifest = pd.read_parquet(MANIFEST_PATH)
OLD_PREFIX = "/kaggle/input/food-freshness-dataset/"
NEW_PREFIX = "/kaggle/input/datasets/ulnnproject/food-freshness-dataset/"
n_translated = int(manifest["path"].str.startswith(OLD_PREFIX).sum())
if n_translated:
    manifest["path"] = manifest["path"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    print(f"Translated {n_translated:,} legacy paths.")
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
print(f"Test split: {len(test_df):,} images.")

## Block 1 — Classifier on ground‑truth full images

In [ ]:
ckpt = torch.load(CLASSIFIER_CKPT, map_location=device, weights_only=True)
assert tuple(ckpt["class_names"]) == COMBINED_CLASSES, "checkpoint class_names mismatch"
image_size = int(ckpt["image_size"])
architecture = ckpt["architecture"]

model = timm.create_model(architecture, pretrained=False, num_classes=NUM_CLASSES)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
eval_transform = create_transform(
    input_size=image_size, is_training=False,
    interpolation="bicubic", crop_pct=0.95,
)

class TestDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row.path).convert("RGB")
        return eval_transform(img), CLASS_TO_INDEX[row.combined_label]

loader = DataLoader(
    TestDataset(test_df), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

all_targets, all_preds = [], []
with torch.no_grad():
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        logits = model(images)
        all_targets.append(targets.numpy())
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
targets_arr = np.concatenate(all_targets)
preds_arr = np.concatenate(all_preds)

classifier_macro_f1 = float(f1_score(targets_arr, preds_arr, average="macro"))
classifier_top1 = float((preds_arr == targets_arr).mean())
classifier_report = classification_report(
    targets_arr, preds_arr,
    labels=list(range(NUM_CLASSES)),
    target_names=list(COMBINED_CLASSES),
    output_dict=True, zero_division=0,
)

print(f"Classifier macro F1: {classifier_macro_f1:.4f}")
print(f"Classifier top-1:    {classifier_top1:.4f}")
for c in COMBINED_CLASSES:
    f1 = classifier_report[c]["f1-score"]
    print(f"  {c:25s} F1={f1:.3f}")

## Block 2 — Detector at multiple operating points

Runs YOLO26s at three confidence thresholds against the held‑out detection test split (the same images that received pseudo‑labels in notebook 2 and were assigned to `split=test`). Pseudo‑labels are not perfect ground truth, so treat absolute numbers with a grain of salt — what matters most is the relative recall improvement vs. the previous detector and the per‑class breakdown.

In [ ]:
# Notebook 2 wrote `path: /kaggle/working/detector` into the YAML � that path doesn't
# exist when this notebook attaches the published dataset. Rewrite the `path:` line to
# point at the actual mount and save a patched copy alongside our outputs.
import re

original_yaml = DETECTOR_DATASET_YAML.read_text()
mounted_root = DETECTOR_DATASET_YAML.parent  # images/ and labels/ are siblings of the yaml

patched_yaml = re.sub(
    r"^path:.*$",
    f"path: {mounted_root}",
    original_yaml,
    count=1,
    flags=re.MULTILINE,
)
patched_path = OUTPUT_DIR / "detector_dataset.yaml"
patched_path.write_text(patched_yaml)
DETECTOR_DATASET_YAML = patched_path

# Sanity-check that the test (and val) image directories actually exist where the
# patched YAML now points.
for split in ("val", "test"):
    p = mounted_root / "images" / split
    n = len(list(p.glob("*.*"))) if p.exists() else 0
    print(f"  {split}: {p} (exists={p.exists()}, images={n})")

print(f"\nPatched YAML written to: {patched_path}")
print(patched_yaml)

In [ ]:
yolo = YOLO(str(DETECTOR_CKPT))
yolo.to(device)

detector_blocks = {}
for conf in DETECTOR_OPERATING_POINTS:
    metrics = yolo.val(
        data=str(DETECTOR_DATASET_YAML), split="test",
        conf=conf, iou=0.45, max_det=MAX_DETECTIONS, verbose=False,
    )
    names = list(yolo.names.values()) if isinstance(yolo.names, dict) else list(yolo.names)
    per_class = {}
    if metrics.box.maps is not None:
        for i, ap in enumerate(metrics.box.maps):
            per_class[names[i]] = {"mAP50_95": float(ap)}
    if metrics.box.ap50 is not None:
        for i, ap50 in enumerate(metrics.box.ap50):
            per_class.setdefault(names[i], {})["mAP50"] = float(ap50)
    if metrics.box.p is not None:
        for i, p in enumerate(metrics.box.p):
            per_class.setdefault(names[i], {})["precision"] = float(p)
    if metrics.box.r is not None:
        for i, r in enumerate(metrics.box.r):
            per_class.setdefault(names[i], {})["recall"] = float(r)
    detector_blocks[f"conf_{conf}"] = {
        "overall": {
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50_95": float(metrics.box.map),
        },
        "per_class": per_class,
    }
    print(
        f"conf={conf}: P={metrics.box.mp:.3f}  R={metrics.box.mr:.3f}  "
        f"mAP50={metrics.box.map50:.3f}  mAP50-95={metrics.box.map:.3f}"
    )

## Block 3 — End‑to‑end pipeline simulation

Replicates the local `FreshnessPipeline.predict` logic: run YOLO26s, take the top detection if any, otherwise run the classifier on the full image, otherwise abstain. We measure joint accuracy on the 24‑class label and the abstention rate (rows where both stages refused to commit).

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def classifier_predict_pil(image: Image.Image) -> tuple[str, float]:
    tensor = eval_transform(image).unsqueeze(0).to(device)
    probs = F.softmax(model(tensor), dim=1)[0]
    idx = int(probs.argmax().item())
    return COMBINED_CLASSES[idx], float(probs[idx].item())

@torch.no_grad()
def detect_top_pil(image: Image.Image, conf: float) -> tuple[str, float] | None:
    arr = np.asarray(image)
    result = yolo.predict(
        source=arr, conf=conf, iou=0.45, max_det=MAX_DETECTIONS, verbose=False,
    )[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None
    confs = result.boxes.conf.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)
    best = int(np.argmax(confs))
    name_dict = result.names if isinstance(result.names, dict) else dict(enumerate(result.names))
    return name_dict[classes[best]], float(confs[best])

predictions: list[dict] = []
for row in test_df.itertuples(index=False):
    try:
        with Image.open(row.path) as img:
            pil = img.convert("RGB")
    except Exception:
        continue
    detection = detect_top_pil(pil, conf=0.25)
    if detection is not None:
        pred_label, pred_conf = detection
        source = "detector"
    else:
        pred_label, pred_conf = classifier_predict_pil(pil)
        if pred_conf < FALLBACK_THRESHOLD:
            pred_label, pred_conf, source = "unknown_n_a", 0.0, "unknown"
        else:
            source = "classifier"
    predictions.append({
        "path": row.path,
        "true_label": row.combined_label,
        "pred_label": pred_label,
        "pred_conf": pred_conf,
        "source": source,
    })

pred_df = pd.DataFrame(predictions)
non_abstain = pred_df[pred_df.source != "unknown"]
joint_accuracy = float((non_abstain.pred_label == non_abstain.true_label).mean())
abstention_rate = float((pred_df.source == "unknown").mean())
source_counts = Counter(pred_df.source)

non_abstain_macro_f1 = float(
    f1_score(non_abstain.true_label, non_abstain.pred_label, average="macro", labels=list(COMBINED_CLASSES))
)

print(f"End-to-end joint accuracy (excl. abstain): {joint_accuracy:.4f}")
print(f"End-to-end macro F1   (excl. abstain): {non_abstain_macro_f1:.4f}")
print(f"Abstention rate:       {abstention_rate:.4f}")
print(f"Sources: {dict(source_counts)}")

## Save eval_report.json + eval_report.md

In [ ]:
report_payload = {
    "classifier_only": {
        "macro_f1": classifier_macro_f1,
        "top1_accuracy": classifier_top1,
        "per_class": {c: classifier_report[c] for c in COMBINED_CLASSES},
    },
    "detector": detector_blocks,
    "end_to_end": {
        "joint_accuracy_excl_abstain": joint_accuracy,
        "macro_f1_excl_abstain": non_abstain_macro_f1,
        "abstention_rate": abstention_rate,
        "source_counts": dict(source_counts),
    },
    "contract": {
        "classes": list(COMBINED_CLASSES),
        "detector_checkpoint": str(DETECTOR_CKPT.name),
        "classifier_checkpoint": str(CLASSIFIER_CKPT.name),
    },
}
(OUTPUT_DIR / "eval_report.json").write_text(json.dumps(report_payload, indent=2))

md = [
    "# FreshGuard End‑to‑End Evaluation Report",
    "",
    "All metrics computed on the cluster‑disjoint **test split** frozen by notebook 1.",
    "",
    "## Classifier‑only (ground‑truth full images, 24 classes)",
    "",
    f"- **Macro F1: {classifier_macro_f1:.4f}**",
    f"- Top‑1 accuracy: {classifier_top1:.4f}",
    "",
    "### Per‑class F1 (sorted ascending — minority classes first)",
    "",
    "| Class | F1 | Precision | Recall | Support |",
    "|---|---:|---:|---:|---:|",
    *[
        f"| {c} | {classifier_report[c]['f1-score']:.3f} | "
        f"{classifier_report[c]['precision']:.3f} | "
        f"{classifier_report[c]['recall']:.3f} | "
        f"{int(classifier_report[c]['support'])} |"
        for c in sorted(COMBINED_CLASSES, key=lambda x: classifier_report[x]["f1-score"])
    ],
    "",
    "## Detector (YOLO26s) at multiple operating points",
    "",
    "| Conf | Precision | Recall | mAP50 | mAP50‑95 |",
    "|---:|---:|---:|---:|---:|",
    *[
        f"| {conf} | {detector_blocks[f'conf_{conf}']['overall']['precision']:.3f} | "
        f"{detector_blocks[f'conf_{conf}']['overall']['recall']:.3f} | "
        f"{detector_blocks[f'conf_{conf}']['overall']['mAP50']:.3f} | "
        f"{detector_blocks[f'conf_{conf}']['overall']['mAP50_95']:.3f} |"
        for conf in DETECTOR_OPERATING_POINTS
    ],
    "",
    "## End‑to‑end pipeline (detector → classifier fallback → unknown)",
    "",
    f"- Joint accuracy (excluding abstentions): **{joint_accuracy:.4f}**",
    f"- Macro F1 (excluding abstentions): **{non_abstain_macro_f1:.4f}**",
    f"- Abstention rate: {abstention_rate:.4f}",
    f"- Source breakdown: {dict(source_counts)}",
    "",
    "## How to use these numbers",
    "",
    "- **Macro F1 is the headline.** Top‑1 accuracy hides minority‑class failure under the 41:1 imbalance — don't quote top‑1 in isolation.",
    "- The end‑to‑end joint accuracy is the user‑visible quality of the local app on test images. The classifier‑only number is the ceiling that perfect detection would deliver.",
    "- The detector‑only block shows the recall/precision tradeoff. Pick a conf threshold for `configs/inference.toml` based on whether your demo prefers misses (high conf) or false positives (low conf).",
]
(OUTPUT_DIR / "eval_report.md").write_text("\n".join(md))
print(f"Wrote: {OUTPUT_DIR / 'eval_report.json'}")
print(f"Wrote: {OUTPUT_DIR / 'eval_report.md'}")

## Next steps

1. Download `eval_report.md` and `eval_report.json` from the notebook output.
2. Paste the markdown summary into the local repo's `src/freshness/ui/content.py` (Metrics section) and update `PRD.md` with the headline numbers.
3. Mirror `yolo26s_food_freshness.pt` and `efficientnetv2s_food_freshness.pt` to a GitHub Release named `v0.2.0` (or whatever version you're cutting). Confirm `python scripts/download_artifacts.py` pulls them.
4. Tag the repo and you're done — local Streamlit demo runs end‑to‑end against the new weights.